In [25]:
import pandas as pd
import numpy as np
from dataclasses import dataclass
from scipy.optimize import least_squares
from scipy.interpolate import PchipInterpolator


In [30]:
path = "/home/poucho/code/etl/data/binance/order_book/BTC/20260616.csv.gz"

In [27]:
path = "/home/poucho/code/etl/data/deribit/options/volatility/BTC_BTC/20260616.csv.gz"

In [31]:
df = pd.read_csv(path)
df.shape
len(df)

(60, 111)

In [32]:
df.head()


,snapshot_time,asset,symbol,bid_price_1,bid_size_1,ask_price_1,ask_size_1,bid_price_2,bid_size_2,ask_price_2,...,ask_price_25,ask_size_25,mid_price,spread,spread_bps,microprice,bid_volume,ask_volume,total_volume,imbalance
0,2026-06-16 16:38:17.203933+00:00,BTC,BTC/USDT,65769.99,6.61172,65770.00,0.42754,65769.98,0.00273,65770.01,...,65775.99,0.00016,65769.995,0.01,0.001520,65769.999393,19.69289,0.60055,20.29344,0.940813
1,2026-06-16 16:39:00.002965+00:00,BTC,BTC/USDT,65830.00,0.02544,65830.01,7.00362,65829.99,0.00064,65830.02,...,65833.99,0.44039,65830.005,0.01,0.001519,65830.000036,0.11511,15.10823,15.22334,-0.984877
2,2026-06-16 16:40:00.000193+00:00,BTC,BTC/USDT,65865.99,8.26382,65866.00,0.06401,65865.98,0.00032,65866.01,...,65871.80,0.00009,65865.995,0.01,0.001518,65865.999923,10.36599,0.16380,10.52979,0.968888
3,2026-06-16 16:41:00.000076+00:00,BTC,BTC/USDT,65804.00,2.12164,65804.01,0.87063,65803.99,0.00088,65804.02,...,65809.16,0.00010,65804.005,0.01,0.001520,65804.007090,2.38405,1.08113,3.46518,0.376004
4,2026-06-16 16:42:00.000230+00:00,BTC,BTC/USDT,65828.00,2.23963,65828.01,1.27273,65827.99,0.00271,65828.02,...,65833.48,0.00008,65828.005,0.01,0.001519,65828.006376,2.89296,2.28745,5.18041,0.116885


In [20]:
def add_time_to_expiry(
    df,
    snapshot_col="snapshot_time",
    expiry_col="expiry",
    out_col="time_to_expiry",
    day_count=365.0,
    drop_nonpositive=True,
):
    """
    Add numerical time-to-expiry in years.

    Assumes snapshot_time and expiry are strings like:
        '2026-06-16 08:00:00+00:00'

    Parameters
    ----------
    df : pd.DataFrame
    snapshot_col : str
        Column containing valuation/snapshot timestamp.
    expiry_col : str
        Column containing option expiry timestamp.
    out_col : str
        Name of output column.
    day_count : float
        Use 365.0 for Act/365, or 365.25 if preferred.
    drop_nonpositive : bool
        If True, removes expired or zero-expiry rows.

    Returns
    -------
    pd.DataFrame
    """
    df = df.copy()

    snapshot_time = pd.to_datetime(df[snapshot_col], utc=True, errors="coerce")
    expiry = pd.to_datetime(df[expiry_col], utc=True, errors="coerce")

    seconds_per_year = day_count * 24.0 * 60.0 * 60.0

    df[out_col] = (expiry - snapshot_time).dt.total_seconds() / seconds_per_year

    # Remove rows where timestamps failed to parse
    df = df[np.isfinite(df[out_col])]

    if drop_nonpositive:
        df = df[df[out_col] > 0.0]

    return df

df = add_time_to_expiry(df)

In [12]:
last_snap_ts = df['snapshot_time'].unique()[-1]

In [13]:
df_single_snap = df[df.snapshot_time == last_snap_ts]

In [14]:
df_single_snap.tail()

,snapshot_time,instrument_id,asset,settlement_currency,expiry,strike,option_type,bid,ask,mark_price,mark_iv,underlying_price,open_interest,volume,time_to_expiry
483663,2026-06-16 08:36:00.000088+00:00,BTC-26MAR27-180000-P,BTC,BTC,2027-03-26 08:00:00+00:00,180000.0,put,1.5345,1.7315,1.626158,0.4930,68622.01,0.0,0.0,0.775274
483664,2026-06-16 08:36:00.000088+00:00,BTC-26MAR27-190000-C,BTC,BTC,2027-03-26 08:00:00+00:00,190000.0,call,0.0024,0.0029,0.002637,0.5027,68622.01,875.6,0.0,0.775274
483665,2026-06-16 08:36:00.000088+00:00,BTC-26MAR27-190000-P,BTC,BTC,2027-03-26 08:00:00+00:00,190000.0,put,1.6750,1.8810,1.771348,0.5027,68622.01,0.9,0.0,0.775274
483666,2026-06-16 08:36:00.000088+00:00,BTC-26MAR27-200000-C,BTC,BTC,2027-03-26 08:00:00+00:00,200000.0,call,0.0019,0.0026,0.002204,0.5114,68622.01,468.2,0.1,0.775274
483667,2026-06-16 08:36:00.000088+00:00,BTC-26MAR27-200000-P,BTC,BTC,2027-03-26 08:00:00+00:00,200000.0,put,1.8155,2.0300,1.916636,0.5114,68622.01,1.0,0.0,0.775274


In [15]:
expiry_0 = df_single_snap.expiry.unique()[0]
expiry_1 = df_single_snap.expiry.unique()[-1]

In [11]:
df_single_snap[df.expiry == expiry_1]

/tmp/ipykernel_3271132/460826423.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_single_snap[df.expiry == expiry_1]


,snapshot_time,instrument_id,asset,settlement_currency,expiry,strike,option_type,bid,ask,mark_price,mark_iv,underlying_price,open_interest,volume,time_to_expiry
288656,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-30000-C,BTC,USDC,2026-09-25 08:00:00+00:00,30000.0,call,NaN,NaN,36837.018100,74.34,67039.98,0.00,0.0,0.276682
288657,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-30000-P,BTC,USDC,2026-09-25 08:00:00+00:00,30000.0,put,10.0,205.0,124.924112,74.34,67039.98,0.00,0.0,0.276682
288658,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-35000-C,BTC,USDC,2026-09-25 08:00:00+00:00,35000.0,call,NaN,NaN,31962.945822,66.64,67039.98,0.00,0.0,0.276682
288659,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-35000-P,BTC,USDC,2026-09-25 08:00:00+00:00,35000.0,put,190.0,230.0,206.996169,66.64,67039.98,4.67,0.0,0.276682
288660,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-40000-C,BTC,USDC,2026-09-25 08:00:00+00:00,40000.0,call,NaN,NaN,27143.247458,60.04,67039.98,0.00,0.0,0.276682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
288771,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-280000-P,BTC,USDC,2026-09-25 08:00:00+00:00,280000.0,put,NaN,NaN,211101.743069,81.74,67039.98,0.00,0.0,0.276682
288772,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-300000-C,BTC,USDC,2026-09-25 08:00:00+00:00,300000.0,call,NaN,NaN,6.619937,85.19,67039.98,0.00,0.0,0.276682
288773,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-300000-P,BTC,USDC,2026-09-25 08:00:00+00:00,300000.0,put,NaN,NaN,230926.320056,85.19,67039.98,0.00,0.0,0.276682
288774,2026-06-16 08:16:00.000082+00:00,BTC_USDC-25SEP26-320000-C,BTC,USDC,2026-09-25 08:00:00+00:00,320000.0,call,NaN,NaN,4.057975,85.42,67039.98,0.00,0.0,0.276682


In [12]:
# ---------- small numerical helpers ----------
def _softplus(x):
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _inv_softplus(y):
    y = np.asarray(y, dtype=float)
    y = np.maximum(y, 1e-12)
    return np.where(y > 30.0, y, np.log(np.expm1(y)))


def _sigmoid(x):
    x = np.asarray(x, dtype=float)
    return np.where(
        x >= 0.0,
        1.0 / (1.0 + np.exp(-x)),
        np.exp(x) / (1.0 + np.exp(x)),
    )


def _logit(p):
    p = np.clip(p, 1e-8, 1.0 - 1e-8)
    return np.log(p / (1.0 - p))


# ---------- SSVI formula ----------

def ssvi_total_variance(k, theta, rho, eta, gamma):
    """
    Power-law SSVI total implied variance.

        phi(theta) = eta * theta**(-gamma)

    Parameters
    ----------
    k : array_like
        Log-forward moneyness log(K / F_T).
    theta : array_like
        ATM total variance for the corresponding maturity.
    rho, eta, gamma : float
        SSVI parameters.

    Returns
    -------
    total variance w(k, theta)
    """
    k = np.asarray(k, dtype=float)
    theta = np.maximum(np.asarray(theta, dtype=float), 1e-16)

    phi = eta * theta ** (-gamma)
    x = phi * k + rho

    return 0.5 * theta * (
        1.0
        + rho * phi * k
        + np.sqrt(x * x + 1.0 - rho * rho)
    )


@dataclass
class SSVISurfaceFit:
    maturities: np.ndarray
    theta: np.ndarray
    rho: float
    eta: float
    gamma: float
    rmse_vol: float
    result: object

    def theta_at(self, T):
        """
        Monotone interpolation of the fitted ATM total variance curve.

        This implementation only allows interpolation inside the fitted
        maturity range. Extrapolation should be handled deliberately.
        """
        T = np.asarray(T, dtype=float)

        if np.any(T < self.maturities[0]) or np.any(T > self.maturities[-1]):
            raise ValueError("T is outside the fitted maturity range.")

        x = np.r_[0.0, self.maturities]
        y = np.r_[0.0, self.theta]

        interp = PchipInterpolator(x, y, extrapolate=False)
        return np.maximum(interp(T), 1e-16)

    def total_variance(self, K, T, F):
        """
        SSVI total variance for strike K, maturity T, forward F.
        """
        K = np.asarray(K, dtype=float)
        T = np.asarray(T, dtype=float)
        F = np.asarray(F, dtype=float)

        k = np.log(K / F)
        theta = self.theta_at(T)

        return ssvi_total_variance(
            k=k,
            theta=theta,
            rho=self.rho,
            eta=self.eta,
            gamma=self.gamma,
        )

    def implied_vol(self, K, T, F):
        """
        SSVI Black implied volatility.
        """
        T = np.asarray(T, dtype=float)
        w = self.total_variance(K=K, T=T, F=F)
        return np.sqrt(np.maximum(w, 0.0) / T)


# ---------- fitter ----------

def _initial_theta(T, k, w):
    """
    Estimate ATM total variance per maturity by interpolation around k=0.
    """
    maturities = np.unique(T)
    theta0 = []

    for Ti in maturities:
        mask = T == Ti
        ki = k[mask]
        wi = w[mask]

        order = np.argsort(ki)
        ki = ki[order]
        wi = wi[order]

        if ki[0] <= 0.0 <= ki[-1] and len(np.unique(ki)) > 1:
            theta = float(np.interp(0.0, ki, wi))
        else:
            theta = float(wi[np.argmin(np.abs(ki))])

        theta0.append(max(theta, 1e-8))

    theta0 = np.asarray(theta0)

    # Calendar-arbitrage-friendly initialization:
    # ATM total variance should be non-decreasing with maturity.
    return np.maximum.accumulate(theta0 + 1e-10 * np.arange(len(theta0)))


def fit_ssvi_surface(
    T,
    K,
    F,
    iv,
    weights=None,
    rho0=-0.6,
    gamma0=0.45,
    max_nfev=20000,
):
    """
    Fit power-law SSVI to a surface of implied volatilities.

    Model:
        w(k, theta) = theta/2 * [
            1 + rho * phi(theta) * k
              + sqrt((phi(theta) * k + rho)^2 + 1 - rho^2)
        ]

        phi(theta) = eta * theta^(-gamma)

    Inputs
    ------
    T : array_like
        Maturities in years.
    K : array_like
        Strikes.
    F : array_like
        Forward for the corresponding maturity.
        It can be one forward per quote or repeated per smile.
    iv : array_like
        Black implied volatility, e.g. 0.20 for 20%.
    weights : array_like, optional
        Optional quote weights. For example, inverse bid/ask width,
        vega weights, or liquidity weights.

    Returns
    -------
    SSVISurfaceFit
    """
    T = np.asarray(T, dtype=float)
    K = np.asarray(K, dtype=float)
    F = np.asarray(F, dtype=float)
    iv = np.asarray(iv, dtype=float)

    valid = (
        np.isfinite(T)
        & np.isfinite(K)
        & np.isfinite(F)
        & np.isfinite(iv)
        & (T > 0.0)
        & (K > 0.0)
        & (F > 0.0)
        & (iv > 0.0)
    )

    if weights is not None:
        weights = np.asarray(weights, dtype=float)
        valid &= np.isfinite(weights) & (weights >= 0.0)

    T = T[valid]
    K = K[valid]
    F = F[valid]
    iv = iv[valid]

    if weights is None:
        weights = np.ones_like(iv)
    else:
        weights = weights[valid]

    maturities, t_index = np.unique(T, return_inverse=True)

    k = np.log(K / F)
    w_mkt = iv * iv * T

    # Initial ATM total variance curve.
    theta_init = _initial_theta(T, k, w_mkt)

    # Parameterize theta_i as cumulative positive increments:
    # theta_1 > 0 and theta_i increasing in maturity.
    increments_init = np.diff(np.r_[0.0, theta_init])
    increments_init = np.maximum(increments_init, 1e-8)
    u_theta0 = _inv_softplus(increments_init)

    # Parameter transforms:
    # rho   = 0.999 * tanh(z_rho)
    # gamma = 0.5 * sigmoid(z_gamma), so gamma in (0, 1/2)
    # eta   = eta_max(theta, rho, gamma) * sigmoid(z_eta)
    rho0 = float(np.clip(rho0, -0.95, 0.95))
    gamma0 = float(np.clip(gamma0, 1e-4, 0.499))

    z_rho0 = np.arctanh(rho0 / 0.999)
    z_gamma0 = _logit(gamma0 / 0.5)

    theta_max0 = theta_init[-1]
    eta_max0 = min(
        4.0 / ((1.0 + abs(rho0)) * theta_max0 ** (1.0 - gamma0)),
        2.0 / np.sqrt(
            (1.0 + abs(rho0)) * theta_max0 ** (1.0 - 2.0 * gamma0)
        ),
    ) * 0.999

    # Crude initial eta from local ATM total-variance skew.
    slopes = []
    for Ti in maturities:
        mask = T == Ti
        if np.sum(mask) >= 3:
            idx_all = np.where(mask)[0]
            near = idx_all[np.argsort(np.abs(k[mask]))[: min(5, np.sum(mask))]]
            try:
                slope = np.polyfit(k[near], w_mkt[near], 1)[0]
                slopes.append(slope)
            except Exception:
                pass

    if slopes and abs(rho0) > 1e-4:
        denom = rho0 * np.median(theta_init ** (1.0 - gamma0))
        eta0 = np.median(slopes) / denom
        eta0 = float(np.clip(eta0, 1e-4 * eta_max0, 0.5 * eta_max0))
    else:
        eta0 = 0.25 * eta_max0

    z_eta0 = _logit(eta0 / eta_max0)

    p0 = np.r_[u_theta0, z_rho0, z_gamma0, z_eta0]

    sqrt_wgt = np.sqrt(weights)

    def unpack(p):
        u_theta = p[: len(maturities)]

        rho = 0.999 * np.tanh(p[-3])
        gamma = 0.5 * _sigmoid(p[-2])

        increments = _softplus(u_theta) + 1e-12
        theta = np.cumsum(increments)

        theta_max = theta[-1]

        # Sufficient SSVI butterfly-arbitrage constraints on the fitted
        # theta range, specialized to phi(theta)=eta*theta^(-gamma).
        eta_max = min(
            4.0 / ((1.0 + abs(rho)) * theta_max ** (1.0 - gamma)),
            2.0 / np.sqrt(
                (1.0 + abs(rho)) * theta_max ** (1.0 - 2.0 * gamma)
            ),
        ) * 0.999

        eta = eta_max * _sigmoid(p[-1])

        return theta, rho, eta, gamma

    def residuals(p):
        theta, rho, eta, gamma = unpack(p)

        w_fit = ssvi_total_variance(
            k=k,
            theta=theta[t_index],
            rho=rho,
            eta=eta,
            gamma=gamma,
        )

        iv_fit = np.sqrt(np.maximum(w_fit, 1e-16) / T)

        # Fit in implied-vol space.
        # For production, you may prefer vega-weighted price errors.
        return sqrt_wgt * (iv_fit - iv)

    res = least_squares(
        residuals,
        p0,
        loss="soft_l1",
        f_scale=0.005,  # robust scale: 50 vol bps
        max_nfev=max_nfev,
    )

    theta, rho, eta, gamma = unpack(res.x)
    rmse_vol = float(np.sqrt(np.mean(residuals(res.x) ** 2)))

    return SSVISurfaceFit(
        maturities=maturities,
        theta=theta,
        rho=float(rho),
        eta=float(eta),
        gamma=float(gamma),
        rmse_vol=rmse_vol,
        result=res,
    )

In [14]:
fit = fit_ssvi_surface(
    T=df_single_snap["time_to_expiry"].to_numpy(),
    K=df_single_snap["strike"].to_numpy(),
    F=df_single_snap["underlying_price"].to_numpy(),
    iv=df_single_snap["mark_iv"].to_numpy(),
)
print("rho   =", fit.rho)
print("eta   =", fit.eta)
print("gamma =", fit.gamma)
print("RMSE vol =", fit.rmse_vol)


rho   = -0.2095036229848719
eta   = 1.3625113879384871
gamma = 0.4872838787165494
RMSE vol = 0.05913368879592219


In [17]:
# Get fitted vols for the original quotes
df_single_snap["ssvi_iv"] = fit.implied_vol(
    K=df_single_snap["strike"].to_numpy(),
    T=df_single_snap["time_to_expiry"].to_numpy(),
    F=df_single_snap["underlying_price"].to_numpy(),
)

In [22]:

# Price/query a point inside the fitted maturity range
K0 = 65000
T0 = 2/365
F0 = 66000.0

vol0 = fit.implied_vol(K=K0, T=T0, F=F0)
print(vol0)

0.3695226451432375


In [23]:
df_single_snap.strike.unique()

array([ 57000.,  58000.,  59000.,  60000.,  61000.,  61500.,  62000.,
        62500.,  63000.,  63500.,  64000.,  64500.,  65000.,  65500.,
        66000.,  66500.,  67000.,  67500.,  68000.,  68500.,  69000.,
        70000.,  71000.,  72000.,  73000.,  50000.,  53000.,  54000.,
        55000.,  56000.,  74000.,  75000.,  76000.,  77000.,  78000.,
        80000.,  82000.,  85000.,  90000.,  20000.,  30000.,  40000.,
        45000.,  52000.,  84000.,  86000.,  88000.,  92000.,  94000.,
        95000., 100000., 105000., 110000., 115000., 120000., 125000.,
       130000., 135000., 140000., 145000., 150000., 155000., 160000.,
       165000., 170000., 180000., 190000., 200000., 210000., 220000.,
       230000., 240000., 250000., 260000., 280000., 300000., 320000.,
       340000., 360000., 380000.,  46000.,  48000.,  79000.,  81000.,
        83000.,  87000.,  89000.,  91000.,  93000.,  96000.,  98000.,
       102000., 104000., 106000., 108000.,  35000., 175000.])

In [24]:
df_single_snap[df_single_snap["strike"] == 65000]

,snapshot_time,instrument_id,asset,settlement_currency,expiry,strike,option_type,bid,ask,mark_price,mark_iv,underlying_price,open_interest,volume,time_to_expiry,ssvi_iv
288224,2026-06-16 08:16:00.000082+00:00,BTC_USDC-17JUN26-65000-C,BTC,USDC,2026-06-17 08:00:00+00:00,65000.0,call,5.0,NaN,1522.426003,36.87,66434.230000,0.00,0.00,0.002709,0.388945
288225,2026-06-16 08:16:00.000082+00:00,BTC_USDC-17JUN26-65000-P,BTC,USDC,2026-06-17 08:00:00+00:00,65000.0,put,65.0,85.0,79.206723,36.87,66434.230000,1.00,1.50,0.002709,0.388945
288272,2026-06-16 08:16:00.000082+00:00,BTC_USDC-18JUN26-65000-C,BTC,USDC,2026-06-18 08:00:00+00:00,65000.0,call,5.0,NaN,1690.604704,39.49,66440.770000,0.00,0.00,0.005449,0.392817
288273,2026-06-16 08:16:00.000082+00:00,BTC_USDC-18JUN26-65000-P,BTC,USDC,2026-06-18 08:00:00+00:00,65000.0,put,230.0,265.0,250.537708,39.49,66438.900000,3.02,3.02,0.005449,0.392717
288324,2026-06-16 08:16:00.000082+00:00,BTC_USDC-19JUN26-65000-C,BTC,USDC,2026-06-19 08:00:00+00:00,65000.0,call,5.0,NaN,1811.009196,38.60,66443.770000,0.47,0.00,0.008189,0.386885
288325,2026-06-16 08:16:00.000082+00:00,BTC_USDC-19JUN26-65000-P,BTC,USDC,2026-06-19 08:00:00+00:00,65000.0,put,355.0,380.0,370.159439,38.60,66443.770000,2.28,2.00,0.008189,0.386885
288380,2026-06-16 08:16:00.000082+00:00,BTC_USDC-20JUN26-65000-C,BTC,USDC,2026-06-20 08:00:00+00:00,65000.0,call,5.0,NaN,1912.806537,37.70,66445.444286,0.00,0.00,0.010928,0.365020
288381,2026-06-16 08:16:00.000082+00:00,BTC_USDC-20JUN26-65000-P,BTC,USDC,2026-06-20 08:00:00+00:00,65000.0,put,450.0,490.0,467.321665,37.70,66445.444286,0.00,0.00,0.010928,0.365020
288438,2026-06-16 08:16:00.000082+00:00,BTC_USDC-26JUN26-65000-C,BTC,USDC,2026-06-26 08:00:00+00:00,65000.0,call,5.0,NaN,2411.784099,36.27,66467.420000,22.23,2.64,0.027367,0.395577
288439,2026-06-16 08:16:00.000082+00:00,BTC_USDC-26JUN26-65000-P,BTC,USDC,2026-06-26 08:00:00+00:00,65000.0,put,915.0,960.0,947.782950,36.27,66467.420000,16.73,0.90,0.027367,0.395577
